# ModernBERT maize MLM adaptation (Kaggle)

Run this notebook on Kaggle with internet enabled and these two notebook inputs attached:

- `/kaggle/input/modernflorabert-base/florabert`
- `/kaggle/input/maize-nam-train-test-dataset/all_seqs_train.txt` and `all_seqs_test.txt`

The notebook copies the required files from those read-only inputs into `/kaggle/working/florabert_runs` before training.

The repository branch, plant-pretrained checkpoint, tokenizer, and current MLM recipe match the Colab maize-MLM notebook. No tokenizer retraining or random ModernBERT initialization is permitted.

Run the W&B cell before training so the repository Trainer does not fail during its first log event.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}


kaggle_input_root = Path(
    os.environ.get('KAGGLE_INPUT_ROOT', '/kaggle/input')
).expanduser()
plant_input_root = Path(
    os.environ.get(
        'FLORABERT_MODERNFLORABERT_INPUT',
        '/kaggle/input/modernflorabert-base/florabert',
    )
).expanduser()
maize_input_root = Path(
    os.environ.get(
        'FLORABERT_MAIZE_INPUT_DIR',
        '/kaggle/input/maize-nam-train-test-dataset',
    )
).expanduser()
run_root = Path(
    os.environ.get('FLORABERT_RUN_ROOT', '/kaggle/working/florabert_runs')
).expanduser()
repo_dir = Path(
    os.environ.get('FLORABERT_REPO_DIR', '/kaggle/working/florabert')
).expanduser()
repo_url = os.environ.get(
    'FLORABERT_REPO_URL',
    'https://github.com/gurveervirk/florabert.git',
)
repo_ref = os.environ.get(
    'FLORABERT_REPO_REF',
    'feat/modernbert-maize-mlm-ablation',
)
expected_commit = os.environ.get('FLORABERT_REPO_COMMIT', '').strip()

if not (repo_dir / '.git').is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f'{repo_dir} exists but is not an empty git checkout'
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--branch', repo_ref, '--depth', '1', repo_url, str(repo_dir)],
        check=True,
    )

actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'],
    cwd=repo_dir,
    text=True,
).strip()
if expected_commit and actual_commit != expected_commit:
    raise RuntimeError(
        f'Unexpected FloraBERT commit {actual_commit}; expected {expected_commit}'
    )

run_root.mkdir(parents=True, exist_ok=True)
data_root = run_root / 'data'
hf_maize_dir = data_root / 'maize-promoter-sequences'
kaggle_workspace = run_root / 'kaggle-modernflorabert-base-v3'
kaggle_workspace.mkdir(parents=True, exist_ok=True)
plant_checkpoint = kaggle_workspace / 'plant-checkpoint-final'
plant_tokenizer = kaggle_workspace / 'modernbert-tokenizer'
maize_lm_output = (
    run_root / 'models' / 'transformer' / 'language-model-modernbert-maize'
)

for path in [data_root, hf_maize_dir, maize_lm_output]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))

print('Kaggle input root:', kaggle_input_root)
print('Plant input root:', plant_input_root)
print('Maize input root:', maize_input_root)
print('Repo:', repo_dir)
print('Repo ref:', repo_ref)
print('Repo commit:', actual_commit)
print('Run root:', run_root)
print('Maize data directory:', hf_maize_dir)
print('Kaggle working artifact directory:', kaggle_workspace)
print('Maize MLM output directory:', maize_lm_output)

Cloning into '/kaggle/working/florabert'...


Kaggle input root: /kaggle/input
Plant input root: /kaggle/input/modernflorabert-base/florabert
Maize input root: /kaggle/input/maize-nam-train-test-dataset
Repo: /kaggle/working/florabert
Repo ref: feat/modernbert-maize-mlm-ablation
Repo commit: b9c35a1d855fa0d911250fc8c334cadf1e30479f
Run root: /kaggle/working/florabert_runs
Maize data directory: /kaggle/working/florabert_runs/data/maize-promoter-sequences
Kaggle working artifact directory: /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3
Maize MLM output directory: /kaggle/working/florabert_runs/models/transformer/language-model-modernbert-maize


Updating files: 100% (393/393), done.


In [2]:
subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(repo_dir / 'requirements.txt'),
    ],
    cwd=repo_dir,
)

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'wandb>=0.19',
    ]
)

import importlib.metadata as importlib_metadata
import torch

for package_name in [
    'torch',
    'transformers',
    'datasets',
    'accelerate',
    'wandb',
]:
    try:
        print(package_name, importlib_metadata.version(package_name))
    except importlib_metadata.PackageNotFoundError:
        print(package_name, 'not found')

cuda_count = torch.cuda.device_count()
print('CUDA device count:', cuda_count)
if cuda_count:
    for device_idx in range(cuda_count):
        print(f'CUDA {device_idx}: {torch.cuda.get_device_name(device_idx)}')
else:
    raise RuntimeError('A Kaggle GPU runtime is required for practical MLM training.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
torch 2.10.0+cu128
transformers 4.57.6
datasets 5.0.0
accelerate 1.13.0
wandb 0.26.1
CUDA device count: 2
CUDA 0: Tesla T4
CUDA 1: Tesla T4


In [3]:
# The model and maize files are supplied as attached Kaggle inputs, so this
# notebook does not need HF or Kaggle download authentication. W&B uses the
# runtime-only secret helper below.
def runtime_secret(name):
    value = os.environ.get(name)
    return value.strip() if value and value.strip() else None


print('Using attached Kaggle inputs; no dataset download is required.')
print('Plant input root:', plant_input_root)
print('Maize input root:', maize_input_root)

Using attached Kaggle inputs; no dataset download is required.
Plant input root: /kaggle/input/modernflorabert-base/florabert
Maize input root: /kaggle/input/maize-nam-train-test-dataset


In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("key")

if not wandb_key:
    raise RuntimeError('No W&B API key was entered.')

os.environ['WANDB_API_KEY'] = wandb_key
os.environ.setdefault(
    'WANDB_PROJECT',
    'florabert-modernbert-maize-ablation',
)
wandb.login(key=wandb_key)
del wandb_key

print('W&B authentication is ready.')
print('W&B project:', os.environ['WANDB_PROJECT'])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gurveervirk-15517 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B authentication is ready.
W&B project: florabert-modernbert-maize-ablation


In [5]:
# Copy the attached, read-only maize files into the writable run directory.
maize_input_files = {
    'all_seqs_train.txt': maize_input_root / 'all_seqs_train.txt',
    'all_seqs_test.txt': maize_input_root / 'all_seqs_test.txt',
}


def count_nonempty_lines(path):
    count = 0
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                count += 1
    return count


def copy_input_file(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    if not source_path.is_file() or source_path.stat().st_size == 0:
        raise FileNotFoundError(
            f'Missing or empty attached maize file: {source_path}'
        )

    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if (
        not destination_path.is_file()
        or destination_path.stat().st_size != source_path.stat().st_size
    ):
        shutil.copy2(source_path, destination_path)
        print('Copied', source_path, '->', destination_path)
    else:
        print('Reusing copied file:', destination_path)


maize_paths = {}
maize_counts = {}
for filename, source_path in maize_input_files.items():
    destination_path = hf_maize_dir / filename
    copy_input_file(source_path, destination_path)

    maize_paths[filename] = destination_path
    maize_counts[filename] = count_nonempty_lines(destination_path)
    if maize_counts[filename] == 0:
        raise ValueError(
            f'Maize file has no non-empty sequences: {destination_path}'
        )

    print(
        filename,
        'source=',
        source_path,
        'working copy=',
        destination_path,
        'bytes=',
        destination_path.stat().st_size,
        'sequences=',
        maize_counts[filename],
    )

assert maize_counts['all_seqs_train.txt'] > 0
assert maize_counts['all_seqs_test.txt'] > 0

Copied /kaggle/input/maize-nam-train-test-dataset/all_seqs_train.txt -> /kaggle/working/florabert_runs/data/maize-promoter-sequences/all_seqs_train.txt
all_seqs_train.txt source= /kaggle/input/maize-nam-train-test-dataset/all_seqs_train.txt working copy= /kaggle/working/florabert_runs/data/maize-promoter-sequences/all_seqs_train.txt bytes= 721544554 sequences= 770688
Copied /kaggle/input/maize-nam-train-test-dataset/all_seqs_test.txt -> /kaggle/working/florabert_runs/data/maize-promoter-sequences/all_seqs_test.txt
all_seqs_test.txt source= /kaggle/input/maize-nam-train-test-dataset/all_seqs_test.txt working copy= /kaggle/working/florabert_runs/data/maize-promoter-sequences/all_seqs_test.txt bytes= 309228967 sequences= 330296


In [6]:
# Copy the plant checkpoint and tokenizer from the attached, read-only
# ModernFloraBERT input into the writable Kaggle working directory.
def has_model_weights(directory):
    return any(
        path.is_file()
        for pattern in (
            '*.safetensors',
            '*.bin',
            '*.safetensors.index.json',
            '*.bin.index.json',
        )
        for path in directory.glob(pattern)
    )


model_candidates = [
    plant_input_root / 'models' / 'transformer' / 'language-model-modernbert'
]
plant_model_source = next(
    (
        candidate
        for candidate in model_candidates
        if (candidate / 'config.json').is_file()
        and has_model_weights(candidate)
    ),
    None,
)

tokenizer_candidates = [
    plant_input_root / 'models' / 'modernbert-byte-level-bpe-tokenizer',
]
plant_tokenizer_source = next(
    (
        candidate
        for candidate in tokenizer_candidates
        if (candidate / 'tokenizer.json').is_file()
    ),
    None,
)

if plant_tokenizer_source is None and plant_input_root.is_dir():
    discovered_tokenizers = sorted(
        tokenizer_path.parent
        for tokenizer_path in plant_input_root.rglob('tokenizer.json')
        if 'modernbert' in str(tokenizer_path).lower()
    )
    plant_tokenizer_source = (
        discovered_tokenizers[0] if discovered_tokenizers else None
    )

if plant_model_source is None:
    raise FileNotFoundError(
        'Could not find a ModernBERT checkpoint under '
        f'{plant_input_root}. Expected config.json plus model weights.'
    )
if plant_tokenizer_source is None:
    raise FileNotFoundError(
        'Could not find tokenizer.json under '
        f'{plant_input_root}.'
    )

print('Plant checkpoint source:', plant_model_source)
print('Plant tokenizer source:', plant_tokenizer_source)

shutil.copytree(
    plant_model_source,
    plant_checkpoint,
    dirs_exist_ok=True,
)
shutil.copytree(
    plant_tokenizer_source,
    plant_tokenizer,
    dirs_exist_ok=True,
)

if not (plant_checkpoint / 'config.json').is_file() or not has_model_weights(plant_checkpoint):
    raise RuntimeError(
        f'Copied plant checkpoint is incomplete: {plant_checkpoint}'
    )
if not (plant_tokenizer / 'tokenizer.json').is_file():
    raise RuntimeError(
        f'Copied plant tokenizer is incomplete: {plant_tokenizer}'
    )

print('Copied plant checkpoint:', plant_checkpoint)
print('Copied plant tokenizer:', plant_tokenizer)

Plant checkpoint source: /kaggle/input/modernflorabert-base/florabert/models/transformer/language-model-modernbert
Plant tokenizer source: /kaggle/input/modernflorabert-base/florabert/models/modernbert-byte-level-bpe-tokenizer
Copied plant checkpoint: /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/plant-checkpoint-final
Copied plant tokenizer: /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/modernbert-tokenizer


In [7]:
from transformers import AutoConfig, PreTrainedTokenizerFast

from module.florabert import config as flora_config
from module.florabert import transformers as flora_transformers
from module.florabert import utils as flora_utils


plant_config = AutoConfig.from_pretrained(
    str(plant_checkpoint),
    local_files_only=True,
)
plant_tokenizer_obj = PreTrainedTokenizerFast.from_pretrained(
    str(plant_tokenizer),
    local_files_only=True,
)

print('Selected pretrained checkpoint:', plant_checkpoint)
print('Selected tokenizer:', plant_tokenizer)
print('Model type:', plant_config.model_type)
print('Architectures:', plant_config.architectures)
print('Checkpoint vocab size:', plant_config.vocab_size)
print('Tokenizer vocab size:', len(plant_tokenizer_obj))
print('Model max positions:', plant_config.max_position_embeddings)

assert plant_config.model_type == 'modernbert'
assert 'ModernBertForMaskedLM' in (plant_config.architectures or [])
assert plant_config.vocab_size == len(plant_tokenizer_obj), (
    'Refusing to resize or retrain the plant tokenizer: checkpoint vocab '
    f'{plant_config.vocab_size} != tokenizer vocab {len(plant_tokenizer_obj)}'
)
assert (plant_checkpoint / 'model.safetensors').is_file()

flora_config.reload_settings()
lm_settings = flora_utils.get_model_settings(
    flora_config.settings,
    model_name='modernbert-lm',
)
expected_positions = lm_settings['max_tokenized_len'] + 2
assert plant_config.max_position_embeddings == expected_positions

_, loaded_tokenizer, plant_lm = flora_transformers.load_model(
    'modernbert-lm',
    str(plant_tokenizer),
    pretrained_model=str(plant_checkpoint),
    **lm_settings,
)

total_params = flora_utils.count_model_parameters(
    plant_lm,
    trainable_only=False,
)
print('Verified plant checkpoint weights loaded.')
print('Loaded parameter count:', total_params)
assert len(loaded_tokenizer) == plant_config.vocab_size

smoke_inputs = loaded_tokenizer(
    'ACGTACGTACGTTTTAAACCCGGG',
    return_tensors='pt',
    max_length=loaded_tokenizer.model_max_length,
    truncation=True,
    padding='max_length',
)
plant_lm.eval()
with torch.no_grad():
    smoke_outputs = plant_lm(**smoke_inputs)

assert smoke_outputs.logits.ndim == 3
assert torch.isfinite(smoke_outputs.logits).all()
print('One MLM forward pass:', tuple(smoke_outputs.logits.shape))

del plant_lm, loaded_tokenizer, smoke_outputs, smoke_inputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Selected pretrained checkpoint: /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/plant-checkpoint-final
Selected tokenizer: /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/modernbert-tokenizer
Model type: modernbert
Architectures: ['ModernBertForMaskedLM']
Checkpoint vocab size: 5000
Tokenizer vocab size: 5000
Model max positions: 258
Loading from pretrained model /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/plant-checkpoint-final
Verified pretrained load: 38 base parameter tensors loaded from /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/plant-checkpoint-final
Verified plant checkpoint weights loaded.
Loaded parameter count: 46912904
One MLM forward pass: (1, 258, 5000)


In [8]:
pretrain_settings = dict(flora_config.settings['training']['pretrain'])
mlm_learning_rate = (
    float(os.environ['FLORABERT_MLM_LEARNING_RATE'])
    if os.environ.get('FLORABERT_MLM_LEARNING_RATE')
    else None
)
mlm_epochs = (
    int(os.environ['FLORABERT_MLM_EPOCHS'])
    if os.environ.get('FLORABERT_MLM_EPOCHS')
    else None
)
n_workers = int(os.environ.get('FLORABERT_DATA_WORKERS', '2'))
force_rerun = env_bool('FLORABERT_FORCE_RERUN', False)
mlm_resume_from = (
    Path(os.environ['FLORABERT_MLM_RESUME_FROM']).expanduser()
    if os.environ.get('FLORABERT_MLM_RESUME_FROM')
    else None
)

print('Current repo MLM settings:', pretrain_settings)
print('Effective learning rate:', mlm_learning_rate or pretrain_settings['learning_rate'])
print('Effective epochs:', mlm_epochs or pretrain_settings['num_train_epochs'])
print('MLM resume checkpoint:', mlm_resume_from or 'none')
print('CUDA devices:', cuda_count)

Current repo MLM settings: {'num_train_epochs': 3, 'per_device_train_batch_size': 128, 'per_device_eval_batch_size': 128, 'fp16': True, 'logging_steps': 50, 'eval_steps': 200, 'save_steps': 400, 'save_total_limit': 20, 'gradient_accumulation_steps': 12, 'learning_rate': 0.0001, 'weight_decay': 0, 'adam_epsilon': 1e-08, 'max_grad_norm': 10, 'warmup_steps': 50, 'optimizer': 'lamb', 'scheduler': 'linear', 'mlm_prob': 0.15}
Effective learning rate: 0.0001
Effective epochs: 3
MLM resume checkpoint: none
CUDA devices: 2


In [9]:
def launch_repo_script(relative_script, arguments):
    script_path = repo_dir / relative_script

    if cuda_count > 1:
        accelerate_exe = shutil.which('accelerate')
        if accelerate_exe:
            command = [
                accelerate_exe,
                'launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
        else:
            command = [
                sys.executable,
                '-m',
                'accelerate.commands.launch',
                '--num_processes',
                str(cuda_count),
                str(script_path),
                *map(str, arguments),
            ]
    else:
        command = [sys.executable, '-u', str(script_path), *map(str, arguments)]

    environment = os.environ.copy()
    environment['PYTHONPATH'] = (
        str(repo_dir) + os.pathsep + environment.get('PYTHONPATH', '')
    )
    environment['PYTHONUNBUFFERED'] = '1'

    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(repo_dir),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

In [10]:
mlm_source = mlm_resume_from or plant_checkpoint
if not (mlm_source / 'config.json').is_file():
    raise FileNotFoundError(
        f'MLM source checkpoint is missing config.json: {mlm_source}'
    )
if not any(
    path.is_file()
    for pattern in ('*.safetensors', '*.bin', '*.safetensors.index.json', '*.bin.index.json')
    for path in mlm_source.glob(pattern)
):
    raise FileNotFoundError(
        f'MLM source checkpoint has no model weights: {mlm_source}'
    )

print('Resolved maize train:', maize_paths['all_seqs_train.txt'])
print('Resolved maize test:', maize_paths['all_seqs_test.txt'])
print('Maize train sequences:', maize_counts['all_seqs_train.txt'])
print('Maize test sequences:', maize_counts['all_seqs_test.txt'])
print('Pretrained MLM checkpoint being loaded:', mlm_source)
print('Maize MLM output:', maize_lm_output)

maize_output_config = maize_lm_output / 'config.json'
if (
    maize_output_config.is_file()
    and not force_rerun
    and mlm_resume_from is None
):
    print(
        'Maize MLM output already exists; set FLORABERT_FORCE_RERUN=1 '
        'to retrain:',
        maize_lm_output,
    )
else:
    mlm_arguments = [
        '--model-name',
        'modernbert-lm',
        '--data-dir',
        str(hf_maize_dir),
        '--train-data',
        'all_seqs_train.txt',
        '--test-data',
        'all_seqs_test.txt',
        '--tokenizer-dir',
        str(plant_tokenizer),
        '--output-dir',
        str(maize_lm_output),
        '--precision',
        'fp16',
        '--n-workers',
        str(n_workers),
        '--pretrained-model',
        str(mlm_source),
    ]

    if mlm_resume_from is not None:
        mlm_arguments.extend(['--resume-from-checkpoint', str(mlm_resume_from)])
    if mlm_learning_rate is not None:
        mlm_arguments.extend(['--learning-rate', str(mlm_learning_rate)])
    if mlm_epochs is not None:
        mlm_arguments.extend(['--num-train-epochs', str(mlm_epochs)])

    launch_repo_script(
        Path('scripts/1-modeling/pretrain.py'),
        mlm_arguments,
    )

Resolved maize train: /kaggle/working/florabert_runs/data/maize-promoter-sequences/all_seqs_train.txt
Resolved maize test: /kaggle/working/florabert_runs/data/maize-promoter-sequences/all_seqs_test.txt
Maize train sequences: 770688
Maize test sequences: 330296
Pretrained MLM checkpoint being loaded: /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/plant-checkpoint-final
Maize MLM output: /kaggle/working/florabert_runs/models/transformer/language-model-modernbert-maize
Running: /usr/local/bin/accelerate launch --num_processes 2 /kaggle/working/florabert/scripts/1-modeling/pretrain.py --model-name modernbert-lm --data-dir /kaggle/working/florabert_runs/data/maize-promoter-sequences --train-data all_seqs_train.txt --test-data all_seqs_test.txt --tokenizer-dir /kaggle/working/florabert_runs/kaggle-modernflorabert-base-v3/modernbert-tokenizer --output-dir /kaggle/working/florabert_runs/models/transformer/language-model-modernbert-maize --precision fp16 --n-workers 2 --pretraine

In [11]:
assert (maize_lm_output / 'config.json').is_file()
assert any(
    path.is_file()
    for pattern in ('*.safetensors', '*.bin', '*.safetensors.index.json', '*.bin.index.json')
    for path in maize_lm_output.glob(pattern)
)

print('Maize-adapted ModernBERT checkpoint is ready:', maize_lm_output)
print('The checkpoint is available under /kaggle/working for downstream use.')

Maize-adapted ModernBERT checkpoint is ready: /kaggle/working/florabert_runs/models/transformer/language-model-modernbert-maize
The checkpoint is available under /kaggle/working for downstream use.
